In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_df = spark.table("workspace.default.silver_dataco")

print(f"Silver loaded: {silver_df.count()} rows x {len(silver_df.columns)} columns")

In [0]:
#KPI 1: Fill Rate (On-Time Delivery Rate)
#Formula: On-time orders / Total orders * 100

kpi_fill_rate = (
    silver_df
    .groupBy("order_year", "order_yearmonth", "order_region")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("is_late").alias("late_orders"),
        F.round(
            (1 - F.sum("is_late") / F.count("order_id")) * 100, 2
        ).alias("fill_rate_pct")
    )
    .orderBy("order_yearmonth", "order_region")
)

display(kpi_fill_rate)

In [0]:
#KPI 2 and 3: Inventory Turnover & Days of Inventory (DOI)
#Formula Turnover: Total Sales / Avg Inventory Value
#Formula DOI: 365 / Inventory Turnover

kpi_inventory = (
    silver_df
    .withColumn("inventory_value", 
                F.col("order_item_product_price") * F.col("order_item_quantity"))
    .groupBy("order_year", "order_yearmonth", "category_name")
    .agg(
        F.round(F.sum("sales"), 2).alias("total_sales"),
        F.round(F.avg("inventory_value"), 2).alias("avg_inventory_value"),
        F.count("order_id").alias("total_orders")
    )
    .withColumn("inventory_turnover",
                F.round(F.col("total_sales") / F.col("avg_inventory_value"), 2))
    .withColumn("doi",
                F.round(F.lit(365) / F.col("inventory_turnover"), 1))
    .orderBy("order_yearmonth", "category_name")
)

display(kpi_inventory)

In [0]:
#KPI 4: Slow Movers
#Logic: Products whose total orders are below 50% of category average

#Orders per product
product_orders = (
    silver_df
    .groupBy("category_name", "product_name")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("sales"), 2).alias("total_sales")
    )
)

#Average orders per category
window_category = Window.partitionBy("category_name")

kpi_slow_movers = (
    product_orders
    .withColumn("avg_orders_category",
                F.round(F.avg("total_orders").over(window_category), 1))
    .withColumn("pct_vs_avg",
                F.round(F.col("total_orders") / F.col("avg_orders_category") * 100, 1))
    .withColumn("is_slow_mover",
                F.when(F.col("pct_vs_avg") < 50, "Yes").otherwise("No"))
    .orderBy("category_name", "total_orders")
)

display(kpi_slow_movers)

In [0]:
#KPI 5: Excess & Obsolete (E&O) - Adjusted Logic
#Logic: High unit price + Low order volume = E&O Risk
#(Proxy for high inventory value + low demand)

window_cat = Window.partitionBy("category_name")

kpi_eo = (
    silver_df
    .groupBy("category_name", "product_name")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("sales"), 2).alias("total_sales"),
        F.round(F.avg("order_item_product_price"), 2).alias("avg_unit_price")
    )
    .withColumn("avg_orders_cat", F.avg("total_orders").over(window_cat))
    .withColumn("avg_price_cat", F.avg("avg_unit_price").over(window_cat))
    .withColumn("eo_flag",
        F.when(
            (F.col("total_orders") < F.col("avg_orders_cat")) &
            (F.col("avg_unit_price") > F.col("avg_price_cat")),
            "E&O Risk"
        ).otherwise("OK")
    )
    .orderBy("eo_flag", "total_orders")
)

display(kpi_eo)

In [0]:
from pyspark.sql.window import Window
#KPI 6: Forecast vs Real Consumption
#Forecast proxy: 3-month moving average of sales
#Accuracy: 1 - |Real - Forecast| / Real * 100

#Monthly sales per category
monthly_sales = (
    silver_df
    .groupBy("category_name", "order_year", "order_yearmonth")
    .agg(F.round(F.sum("sales"), 2).alias("real_sales"))
)

#3-month moving average as forecast proxy
window_forecast = (
    Window
    .partitionBy("category_name")
    .orderBy("order_yearmonth")
    .rowsBetween(-3, -1)
)

kpi_forecast = (
    monthly_sales
    .withColumn("forecast_sales",
                F.round(F.avg("real_sales").over(window_forecast), 2))
    .withColumn("forecast_accuracy_pct",
                F.round(
                    (1 - F.abs(F.col("real_sales") - F.col("forecast_sales")) 
                    / F.col("real_sales")) * 100, 1))
    .withColumn("variance_pct",
                F.round(
                    (F.col("real_sales") - F.col("forecast_sales")) 
                    / F.col("forecast_sales") * 100, 1))
    .filter(F.col("forecast_sales").isNotNull())
    .orderBy("category_name", "order_yearmonth")
)

display(kpi_forecast)

In [0]:
#KPI 1: Fill Rate
kpi_fill_rate.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_kpi_fill_rate")

#KPI 2 & 3: Inventory Turnover & DOI
kpi_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_kpi_inventory")

#KPI 4: Slow Movers
kpi_slow_movers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_kpi_slow_movers")

#KPI 5: E&O
kpi_eo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_kpi_eo")

#KPI 6: Forecast vs Real
kpi_forecast.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_kpi_forecast")